Here is a breakdown of when and why to use these commands, based on the sources provided:

### **System & Process Monitoring**
*   **`ps aux`**: Use this to **list every running process** on the system to identify what programs are active and who owns them.
*   **`top`**: Use this for **real-time performance monitoring** to see which processes are currently consuming the most CPU or memory.
*   **`free -h`**: Use this to check the system's **total, used, and available memory** in a human-readable format.

### **Networking & Troubleshooting**
*   **`ss -tulpn`**: Use this to **identify open or listening ports** and see exactly which programs are tied to those network connections.
*   **`ip addr`**: Use this to **view or manage the IP addresses** assigned to your various network interfaces.
*   **`ip route`**: Use this to **inspect the routing table** to determine where network traffic is being directed on your system.
*   **`curl -v`**: Use this to transfer data from a URL with **verbose output**, which is essential for troubleshooting HTTP request and response headers.

***

**Note:** The following command is not detailed in the provided sources, so I am providing this information based on general technical knowledge for your interview:

*   **`openssl s_client`**: Use this to **test and debug SSL/TLS connections** to a server. It allows you to view the server's certificate chain and verify that a secure connection can be established before checking higher-level application data.

ps aux | grep app.py | grep -v grep
ss -tulpn | grep -E '5000|5001|:80|:443'
ip addr
ip route
curl -i http://127.0.0.1/
curl -i http://127.0.0.1:5000/health
curl -i http://127.0.0.1:5001/upstream-ok
curl -vk https://127.0.0.1/
openssl s_client -connect 127.0.0.1:443

I’d go layer by layer, from **host → process → network → nginx → app → logs**.

1. **Know the VM first**

   * `hostnamectl`, `uname -a`, `cat /etc/os-release`, `ls -l `, `id <service-user>`
   * Check CPU, memory, disk: `top`, `free -h`, `df -h`

2. **See what is running**

   * `ps aux --sort=-%cpu`
   * `systemctl list-units --type=service --state=running`
   * Find app/nginx related services

3. **Check listening ports**

   * `ss -tulpn`
   * Tells me what ports are open and which process owns them

4. **Inspect nginx**

   * `nginx -t`
   * Look at configs: `/etc/nginx/nginx.conf`, `sites-enabled/`, `conf.d/`
   * Identify server blocks, upstreams, proxy_pass, SSL, root paths

5. **Test traffic path**

   * `curl -iv http://127.0.0.1`
   * `curl -iv http://127.0.0.1:<app_port>`
   * Helps separate **nginx problem vs app problem**

6. **Check logs**

   * Nginx: `/var/log/nginx/access.log`, `error.log`
   * Systemd app: `journalctl -u <service> -xe`
   * App logs if custom location exists

7. **Understand app startup**

   * `systemctl cat <service>`
   * See `ExecStart`, env files, working directory, user, restart policy

8. **Check dependencies**

   * DB, Redis, APIs, DNS, certificates, mounted volumes
   * Use config files and env vars to discover what the app depends on

9. **Validate file and permission issues**

   * App directory, config files, certs, sockets
   * `ls -l`, `id <service-user>`

10. **For future troubleshooting, document**

* Ports
* Service names
* Config locations
* Log locations
* Startup command
* Dependencies
* Common failure points

My mental model is:
**Can I reach nginx? Can nginx reach app? Can app reach dependencies?**
That usually gets you to root cause fast.


>>ps aux --sort=-%cpu 
>>top
(for both above notice PID, COMMAND), for top also notice S:R,S,D,Z,I as well as TIME+

---> ps -fp <PID>

---
>> free -h
manojcloudvm@instance-20260404-024439:~$ free -h
               total        used        free      shared  buff/cache   available
Mem:           3.8Gi       680Mi       2.4Gi       616Ki       1.1Gi       3.2Gi
Swap:             0B          0B          0B

total = total RAM in the VM
used = memory currently used by processes + some kernel usage
free = RAM doing literally nothing right now
shared = memory used by shared areas, often small; not the main thing to focus on
buff/cache = RAM Linux is using for file cache and buffers to speed things up
available = best practical number for “how much memory can I still use without trouble”

Because Linux knows a lot of that 1.1 GiB cache can be freed if needed.
So:

available ≈ free + reclaimable cache/buffer

Swap:
Swap is disk space used as overflow memory when RAM gets tight.
So if physical RAM is full, Linux can move some less-active memory pages from RAM to swap, freeing RAM for active work.

Why it matters in resource contention:
Helps avoid immediate crashes when memory pressure rises
But it is much slower than RAM because disk is far slower
If the system starts swapping heavily, apps can become very slow
In bad cases, you get thrashing: system spends more time moving memory pages in/out than doing real work

In your output:
Swap: 0B 0B 0B
Means no swap is configured

What that means practically:
Good: no swap slowness
Risk: under heavy memory pressure, system has less safety cushion
Then Linux may trigger the OOM killer and kill processes

Interview-safe line:
Swap is disk-backed emergency memory. It can protect against OOM briefly, but if actively used, it usually signals memory contention and severe performance degradation.


------


df -h is mainly for filesystem usage: total size, used, available, and mount point.

>> manojcloudvm@instance-20260404-024439:~$ df -h
Filesystem      Size  Used Avail Use% Mounted on
udev            2.0G     0  2.0G   0% /dev
tmpfs           393M  580K  392M   1% /run
/dev/sda1       9.7G  3.9G  5.4G  42% /
tmpfs           2.0G   32K  2.0G   1% /dev/shm
tmpfs           5.0M     0  5.0M   0% /run/lock
/dev/sda15      124M   12M  113M  10% /boot/efi
tmpfs           393M     0  393M   0% /run/user/1000


>> sudo du -sh /
df tells me which mounted filesystem is filling up; if root /dev/sda1 is the one growing, I switch to du to find which directories on that filesystem are responsible. (use "Mounted on")

>>systemctl list-units --type=service --state=running
>>sudo systemctl status nginx

**Service-managed** means the process is started, stopped, monitored, and often restarted by **systemd**.

Common service-managed things:

* `nginx`
* `docker`
* `sshd`
* `cron`
* `postgresql`
* `redis`
* `kubelet`

You usually control them with:

```bash
systemctl status <service>
systemctl start <service>
systemctl stop <service>
systemctl restart <service>
```

Not service-managed usually means:

* started manually in shell
* `python app.py`
* `node server.js`
* `nohup ... &`
* `screen` / `tmux`

Simple distinction:

* **service-managed** = OS manages lifecycle
* **regular process** = user/script started it directly


manojcloudvm@instance-20260404-024439:~$ ss -tulpn
Netid          State           Recv-Q          Send-Q                     Local Address:Port                      Peer Address:Port          Process                                                                                                                                      
udp            UNCONN          0               0                                0.0.0.0:5355                           0.0.0.0:*                                                                                                                                                          
udp            UNCONN          0               0                             127.0.0.54:53                             0.0.0.0:*                                                                                                                                                          
udp            UNCONN          0               0                          127.0.0.53%lo:53                             0.0.0.0:*                                                                                                                                                          
udp            UNCONN          0               0                        10.128.0.2%ens4:68                             0.0.0.0:*                                                                                                                                                          
udp            UNCONN          0               0                                   [::]:5355                              [::]:*                                                                                                                                                          
tcp            LISTEN          0               4096                       127.0.0.53%lo:53                             0.0.0.0:*                                                                                                                                                          
tcp            LISTEN          0               511                              0.0.0.0:443                            0.0.0.0:*                                                                                                                                                          
tcp            LISTEN          0               511                              0.0.0.0:80                             0.0.0.0:*                                                                                                                                                          
tcp            LISTEN          0               128                              0.0.0.0:22                             0.0.0.0:*                                                                                                                                                          
tcp            LISTEN          0               4096                          127.0.0.54:53                             0.0.0.0:*                                                                                                                                                          
tcp            LISTEN          0               128                              0.0.0.0:20202                          0.0.0.0:*                                                                                                                                                          
tcp            LISTEN          0               128                            127.0.0.1:5001                           0.0.0.0:*              users:(("python",pid=88433,fd=6),("python",pid=88433,fd=5),("python",pid=88432,fd=5))                                                       
tcp            LISTEN          0               128                            127.0.0.1:5000                           0.0.0.0:*              users:(("python",pid=99246,fd=6),("python",pid=99246,fd=5),("python",pid=6953,fd=5))                                                        
tcp            LISTEN          0               20                             127.0.0.1:25                             0.0.0.0:*                                                                                                                                                          
tcp            LISTEN          0               4096                             0.0.0.0:5355                           0.0.0.0:*                                                                                                                                                          
tcp            LISTEN          0               511                                 [::]:443                               [::]:*                                                                                                                                                          
tcp            LISTEN          0               20                                 [::1]:25                                [::]:*                                                                                                                                                          
tcp            LISTEN          0               511                                 [::]:80                                [::]:*                                                                                                                                                          
tcp            LISTEN          0               128                                 [::]:22                                [::]:*                                                                                                                                                          
tcp            LISTEN          0               4096                                   *:20201                                *:*                                                                                                                                                          
tcp            LISTEN          0               4096                                [::]:5355                              [::]:*  


-------------


Yes — with `ss -tulpn`, you mainly care about:

1. **What is listening publicly on `0.0.0.0` or `[::]`**
2. **What is only local on `127.0.0.1`**
3. **Whether the expected process owns the expected port**

From your output :

### Ports you care about most

* **80 / 443** → nginx public web traffic
* **5000 / 5001** → your Python apps, but only on `127.0.0.1`, so internal only
* **22** → SSH, normal for remote access

### Others that are normal / lower priority

* **53** on `127.0.0.53` / `127.0.0.54`
  DNS resolver on the VM, normal
* **68/udp** on `10.128.0.2`
  DHCP client, normal
* **25** on `127.0.0.1` / `[::1]`
  Local mail service, usually not important for your app unless mail is involved
* **5355**
  Name resolution related service, usually not your main concern
* **20201 / 20202**
  These are the ones I’d want to identify if I don’t already know them

### Biggest practical distinction

* **`0.0.0.0:<port>` or `[::]:<port>`** = reachable from outside, more important
* **`127.0.0.1:<port>`** = local only, usually backend/internal service

So here:

* `80`, `443`, `22`, `20202`, `20201` are more security/troubleshooting relevant
* `5000`, `5001` are app backend ports, also important
* DNS/DHCP/mail/helper ports are usually background OS services

### What I would do next for unknown ones

```bash
sudo ss -tulpn | grep 20202
sudo ss -tulpn | grep 20201
ps -fp <PID>
```

Interview-safe line:
**In `ss -tulpn`, I focus first on unexpected public listeners, then map each important port to its owning process, and verify whether it should be public or only local.**


5. Test traffic path

ping checks network reachability using ICMP
curl checks application/service response using protocols like HTTP/HTTPS

Simple difference
ping = “Can I reach that host at network level?”
curl = “Is the web service/API actually responding?”

Example
Host may not reply to ping but website still works
Host may reply to ping but app can still be broken, and curl fails

In troubleshooting
Use ping for basic connectivity
Use curl for real web/app testing

---
Because you are testing **two different layers** separately.

### 1) Test nginx entrypoint
```bash
curl -iv http://127.0.0.1
```

This hits **port 80**, so request goes to **nginx** first.

If this fails:

* nginx may be down
* nginx config may be wrong
* nginx cannot proxy correctly

### 2) Test app directly
```bash
curl -iv http://127.0.0.1:<app_port>
```

This bypasses nginx and talks to the **app itself**.

If this works, app is alive.
---
### How it separates the issue
#### Case A

* `curl http://127.0.0.1` → fails
* `curl http://127.0.0.1:5000` → works

That usually means:
**app is fine, nginx layer is the problem**

#### Case B
* `curl http://127.0.0.1` → fails
* `curl http://127.0.0.1:5000` → also fails

That usually means:
**app itself is down/broken**
or listening on wrong port/interface

#### Case C
* both work
Then problem may be:
* external firewall
* DNS
* TLS/443 only
* load balancer/security group outside VM

Interview-safe line:
**Testing nginx on port 80 and the app directly on its backend port lets me isolate whether the break is in the reverse proxy layer or in the application itself.**

------

manojcloudvm@instance-20260404-024439:~$ curl -iv http://127.0.0.1
*   Trying 127.0.0.1:80...
* Connected to 127.0.0.1 (127.0.0.1) port 80 (#0)
> GET / HTTP/1.1
> Host: 127.0.0.1
> User-Agent: curl/7.88.1
> Accept: */*
> 
< HTTP/1.1 200 OK
HTTP/1.1 200 OK
< Server: nginx/1.22.1
Server: nginx/1.22.1
< Date: Mon, 06 Apr 2026 13:41:28 GMT
Date: Mon, 06 Apr 2026 13:41:28 GMT
< Content-Type: application/json
Content-Type: application/json
< Content-Length: 50
Content-Length: 50
< Connection: keep-alive
Connection: keep-alive

< 
{
  "message": "app1 is running",
  "path": "/"
}
* Connection #0 to host 127.0.0.1 left intact
manojcloudvm@instance-20260404-024439:~$ curl -iv http://127.0.0.1:5000
*   Trying 127.0.0.1:5000...
* Connected to 127.0.0.1 (127.0.0.1) port 5000 (#0)
> GET / HTTP/1.1
> Host: 127.0.0.1:5000
> User-Agent: curl/7.88.1
> Accept: */*
> 
< HTTP/1.1 200 OK
HTTP/1.1 200 OK
< Server: Werkzeug/3.1.8 Python/3.11.2
Server: Werkzeug/3.1.8 Python/3.11.2
< Date: Mon, 06 Apr 2026 13:41:44 GMT
Date: Mon, 06 Apr 2026 13:41:44 GMT
< Content-Type: application/json
Content-Type: application/json
< Content-Length: 50
Content-Length: 50
< Connection: close
Connection: close

< 
{
  "message": "app1 is running",
  "path": "/"
}
* Closing connection 0

ALL ABOUT NGINX:

nginx -t
systemctl status nginx
journalctl -u nginx

systemctl status nginx
systemctl restart nginx
systemctl reload nginx


Yes — as a Linux admin, these are the main nginx things to know:

### 1) Main config areas

* **`/etc/nginx/nginx.conf`**
  Main global config
* **`/etc/nginx/sites-available/`**
  Where site configs are usually stored
* **`/etc/nginx/sites-enabled/`**
  Usually symlinks to the active site configs
* **`/etc/nginx/conf.d/`**
  Extra config snippets, sometimes used instead of sites-available/sites-enabled

### 2) What is `sites-enabled`?

It usually contains **enabled/active virtual hosts**.

Common pattern:

* write config in `sites-available/myapp`
* enable it with symlink into `sites-enabled/`

So nginx reads `sites-enabled`, and that tells you **what site configs are actually active**.

Useful command:

```bash
ls -l /etc/nginx/sites-enabled/
```

### 3) Where to check logs

Common defaults:

* **Access log:** `/var/log/nginx/access.log`
* **Error log:** `/var/log/nginx/error.log`

But many setups use custom logs inside site config, so check:

```bash
grep -R "access_log\|error_log" /etc/nginx
```

### 4) What logs tell you

* **access log** = requests came in, status codes, client IP, path
* **error log** = config issues, upstream failures, permission issues, startup/reload errors

### 5) Admin troubleshooting flow

I usually check in this order: If I am brand new to the VM, my nginx discovery flow is:

* nginx -t
* systemctl status nginx
* cat /etc/nginx/nginx.conf
* ls -l /etc/nginx/sites-enabled/
* cat /etc/nginx/sites-enabled/app1
- which port nginx listens on
- whether it serves static files or proxies to backend
- whether it uses proxy_pass
- whether SSL is configured
- where access/error logs go
* grep -R "access_log\|error_log" /etc/nginx

server {
    listen 80;
    listen [::]:80;
    server_name _;

    access_log /var/log/nginx/app1_access.log;
    error_log /var/log/nginx/app1_error.log;

    location / {
        proxy_pass http://127.0.0.1:5000;
        proxy_http_version 1.1;

        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }
}

server {
    listen 443 ssl;---------------------------------------------------> 1
    listen [::]:443 ssl;---------------------------------------------------> 1
    server_name _; ---------------------------------------------------> 2

    ssl_certificate /etc/nginx/ssl/app1.crt;-------------------------->3
    ssl_certificate_key /etc/nginx/ssl/app1.key; --------------------->3

    access_log /var/log/nginx/app1_ssl_access.log; ------------------->4
    error_log /var/log/nginx/app1_ssl_error.log;---------------------->5

    location / {------------------------------------------------------>6
        proxy_pass http://127.0.0.1:5000; ----------------------------->7
        proxy_http_version 1.1;

        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }
}


### 6) What to care about inside config

* 1. `listen`
* 2. `server_name`
* 3. `ssl cert and key (if exists) `
* 6. `location`
* _. `root`
* 7. `proxy_pass`
* 4. `access_log`
* 5. `error_log`
* SSL cert/key paths
***
Interview-safe line:
**`sites-enabled` shows me which virtual host configs are active, and I use the server block plus access/error logs to understand how nginx is routing traffic and where requests are failing.**


> systemctl cat nginx
manojcloudvm@instance-20260404-024439:~$ systemctl cat nginx
# /lib/systemd/system/nginx.service
# Stop dance for nginx
# =======================
#
# ExecStop sends SIGQUIT (graceful stop) to the nginx process.
# If, after 5s (--retry QUIT/5) nginx is still running, systemd takes control
# and sends SIGTERM (fast shutdown) to the main process.
# After another 5s (TimeoutStopSec=5), and if nginx is alive, systemd sends
# SIGKILL to all the remaining processes in the process group (KillMode=mixed).
#
# nginx signals reference doc:
# http://nginx.org/en/docs/control.html
#
[Unit]
Description=A high performance web server and a reverse proxy server
Documentation=man:nginx(8)
After=network-online.target remote-fs.target nss-lookup.target
Wants=network-online.target

[Service]
Type=forking
PIDFile=/run/nginx.pid
ExecStartPre=/usr/sbin/nginx -t -q -g 'daemon on; master_process on;'
ExecStart=/usr/sbin/nginx -g 'daemon on; master_process on;' ------------------------------------------> * + look for env files location
ExecReload=/usr/sbin/nginx -g 'daemon on; master_process on;' -s reload
ExecStop=-/sbin/start-stop-daemon --quiet --stop --retry QUIT/5 --pidfile /run/nginx.pid
TimeoutStopSec=5
KillMode=mixed

[Install]
WantedBy=multi-user.target